# MLP Fusion vs Attention Fusion Ablation Figures

This notebook generates self-contained supplementary figures for the MFMC fusion-head ablation. It uses only verified summary values and does not depend on the original training data or experiment notebooks.

In [ ]:
# Imports and output directory
from pathlib import Path
import csv

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

EXPERIMENT_ROOT = Path('/home/zhengdeyang/TAFFC_MFMC/MFMC/Supplement/Fusion_Ablation')
FIGURES_DIR = EXPERIMENT_ROOT / 'figures'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Clean, publication-oriented matplotlib defaults.
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.bbox': 'tight',
})

# Low-saturation, print-friendly colors suitable for academic figures.
COLORS = {
    'MLP Fusion': '#1F4E79',
    'Attention Fusion v1': '#8A3B4E',
    'Attention Fusion v2': '#B08D3C',
}

def save_figure(fig, stem):
    """Save a figure as both PDF and high-resolution PNG."""
    pdf_path = FIGURES_DIR / f'{stem}.pdf'
    png_path = FIGURES_DIR / f'{stem}.png'
    fig.savefig(pdf_path)
    fig.savefig(png_path, dpi=300)
    return pdf_path, png_path

In [ ]:
# Verified ablation data
parameter_counts = [
    {'Fusion head': 'MLP Fusion', 'Params per head': 198528, 'Params for three heads': 595584},
    {'Fusion head': 'Attention Fusion v1', 'Params per head': 1710976, 'Params for three heads': 5132928},
    {'Fusion head': 'Attention Fusion v2', 'Params per head': 495744, 'Params for three heads': 1487232},
]

accuracy_results = [
    {
        'Dataset': 'CEAP', 'Protocol': 'Subject dependent', 'Downstream modality': 'EDA',
        'MLP Fusion': 0.868, 'Attention Fusion v1': 0.3227, 'Attention Fusion v2': 0.4992,
        'MLP Fusion std': 0.003, 'Attention Fusion v1 std': 0.0082, 'Attention Fusion v2 std': 0.0218,
    },
    {
        'Dataset': 'CEAP', 'Protocol': 'Subject independent', 'Downstream modality': 'EDA',
        'MLP Fusion': 0.331, 'Attention Fusion v1': 0.2485, 'Attention Fusion v2': 0.2535,
        'MLP Fusion std': 0.020, 'Attention Fusion v1 std': 0.0229, 'Attention Fusion v2 std': 0.0163,
    },
    {
        'Dataset': 'DEAP', 'Protocol': 'Subject dependent', 'Downstream modality': 'EEG',
        'MLP Fusion': 0.987, 'Attention Fusion v1': 0.9227, 'Attention Fusion v2': 0.9244,
        'MLP Fusion std': 0.009, 'Attention Fusion v1 std': 0.0200, 'Attention Fusion v2 std': 0.0215,
    },
    {
        'Dataset': 'DEAP', 'Protocol': 'Subject independent', 'Downstream modality': 'EEG',
        'MLP Fusion': 0.346, 'Attention Fusion v1': 0.2219, 'Attention Fusion v2': 0.3140,
        'MLP Fusion std': 0.030, 'Attention Fusion v1 std': 0.0274, 'Attention Fusion v2 std': 0.0220,
    },
]

fusion_heads = ['MLP Fusion', 'Attention Fusion v1', 'Attention Fusion v2']

In [ ]:
# Optional CSV exports for reproducibility
parameter_csv = RESULTS_DIR / 'fusion_parameter_counts.csv'
with parameter_csv.open('w', newline='') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['Fusion head', 'Params per head', 'Params for three heads', 'Params for three heads (M)'],
    )
    writer.writeheader()
    for row in parameter_counts:
        out = dict(row)
        out['Params for three heads (M)'] = row['Params for three heads'] / 1_000_000
        writer.writerow(out)

accuracy_csv = RESULTS_DIR / 'fusion_ablation_accuracy.csv'
with accuracy_csv.open('w', newline='') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            'Dataset', 'Protocol', 'Downstream modality',
            'MLP Fusion', 'MLP Fusion std',
            'Attention Fusion v1', 'Attention Fusion v1 std',
            'Attention Fusion v2', 'Attention Fusion v2 std',
        ],
    )
    writer.writeheader()
    writer.writerows(accuracy_results)

In [ ]:
# Figure 1: total parameter count comparison for the three fusion heads
labels = [row['Fusion head'] for row in parameter_counts]
values_m = [row['Params for three heads'] / 1_000_000 for row in parameter_counts]

fig, ax = plt.subplots(figsize=(5.0, 3.3))
bars = ax.bar(labels, values_m, color=[COLORS[label] for label in labels], width=0.62, edgecolor='black', linewidth=0.6)

ax.set_ylabel('Parameters (millions)')
ax.set_ylim(0, max(values_m) * 1.18)
ax.grid(axis='y', color='#D9D9D9', alpha=0.8)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', rotation=15)

for bar, value in zip(bars, values_m):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(values_m) * 0.025,
        f'{value:.3f}M',
        ha='center', va='bottom', fontsize=9,
    )

fig.tight_layout()
param_pdf, param_png = save_figure(fig, 'fusion_parameter_count')
plt.show()

In [ ]:
# Figure 2: grouped accuracy comparison with standard-deviation error bars
group_labels = [
    f"{row['Dataset']}\nSubj.-Dep." if row['Protocol'] == 'Subject dependent' else f"{row['Dataset']}\nSubj.-Indep."
    for row in accuracy_results
]
x = list(range(len(group_labels)))
bar_width = 0.20
offsets = [-bar_width, 0, bar_width]

fig, ax = plt.subplots(figsize=(6.2, 3.7))

for head, offset in zip(fusion_heads, offsets):
    means = [row[head] for row in accuracy_results]
    stds = [row[f'{head} std'] for row in accuracy_results]
    positions = [xi + offset for xi in x]
    bars = ax.bar(
        positions,
        means,
        width=bar_width,
        yerr=stds,
        capsize=3,
        color=COLORS[head],
        edgecolor='black',
        linewidth=0.5,
        error_kw={'elinewidth': 0.8, 'capthick': 0.8},
        label=head,
    )
    for bar, mean, std in zip(bars, means, stds):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            mean + std + 0.018,
            f'{mean:.3f}',
            ha='center', va='bottom', fontsize=7.5, rotation=90,
        )

ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.16)
ax.set_xticks(x)
ax.set_xticklabels(group_labels)
ax.grid(axis='y', color='#D9D9D9', alpha=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.14))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
accuracy_pdf, accuracy_png = save_figure(fig, 'fusion_ablation_accuracy')
plt.show()

In [ ]:
# Figure 3: self-contained architecture schematic using matplotlib patches
def draw_architecture_panel(ax, title, steps, color):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.text(0.5, 0.98, title, ha='center', va='top', fontsize=12, fontweight='bold')

    box_x = 0.12
    box_w = 0.76
    box_h = 0.058
    top_y = 0.845
    bottom_y = 0.035
    step_gap = (top_y - bottom_y - box_h) / (len(steps) - 1)

    centers = []
    for idx, label in enumerate(steps):
        y = top_y - idx * step_gap
        centers.append((0.5, y + box_h / 2))
        patch = FancyBboxPatch(
            (box_x, y), box_w, box_h,
            boxstyle='round,pad=0.012,rounding_size=0.018',
            linewidth=0.8,
            edgecolor=color,
            facecolor='white',
        )
        ax.add_patch(patch)
        ax.text(0.5, y + box_h / 2, label, ha='center', va='center', fontsize=8.7, wrap=True)

    for (x0, y0), (x1, y1) in zip(centers[:-1], centers[1:]):
        arrow = FancyArrowPatch(
            (x0, y0 - box_h / 2 - 0.006),
            (x1, y1 + box_h / 2 + 0.006),
            arrowstyle='-|>',
            mutation_scale=9,
            linewidth=0.8,
            color='#555555',
        )
        ax.add_patch(arrow)


mlp_steps = [
    'Input: concat(e_i, e_j), 256D',
    'Linear 256 to 512',
    'BN + ReLU',
    'Linear 512 to 128',
    'BN',
    'Output: e_ij, 128D',
]

attn_v1_steps = [
    'Input: concat(e_i, e_j), 256D',
    'Reshape to two 128D tokens',
    'Linear 128 to 512',
    '4-head self attention + FFN',
    'Mean pooling',
    'Linear 512 to 128',
    'Output: e_ij, 128D',
]

attn_v2_steps = [
    'Input: concat(e_i, e_j), 256D',
    'Reshape to two 128D tokens',
    'Linear 128 to 256',
    'Add modality embedding',
    '4-head self attention + FFN',
    'Flatten two tokens',
    'Linear 512 to 128',
    'Output: e_ij, 128D',
]

fig, axes = plt.subplots(1, 3, figsize=(10.8, 5.4))
draw_architecture_panel(axes[0], 'A. MLP Fusion', mlp_steps, COLORS['MLP Fusion'])
draw_architecture_panel(axes[1], 'B. Attention Fusion v1', attn_v1_steps, COLORS['Attention Fusion v1'])
draw_architecture_panel(axes[2], 'C. Attention Fusion v2', attn_v2_steps, COLORS['Attention Fusion v2'])

fig.tight_layout(w_pad=1.4)
arch_pdf, arch_png = save_figure(fig, 'fusion_head_architecture')
plt.show()

In [ ]:
# Final output manifest
saved_paths = [
    param_pdf, param_png,
    accuracy_pdf, accuracy_png,
    arch_pdf, arch_png,
    parameter_csv, accuracy_csv,
]

print('Saved files:')
for path in saved_paths:
    print(path)